# 01 — Preprocessing Exploration

This notebook demonstrates the **deskew** and **document cropping** logic used
in the MedVault OCR pipeline's preprocessing stage.

The preprocessing module (`backend/preprocessing/pipeline.py`) applies the
following transformations in order:

1. **Document detection & crop** — Edge detection + contour finding to isolate
   the document from the background.
2. **Contrast enhancement** — CLAHE (Contrast Limited Adaptive Histogram
   Equalization) to improve text legibility.
3. **DPI normalization** — Rescale the image to a standard 300 DPI.
4. **Deskew** — Detect and correct the skew angle using the `deskew` library.

We'll walk through each step visually using a sample image.

## Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Ensure the backend package is importable from the notebook
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from backend.preprocessing.pipeline import preprocess
from backend.image_processing import (
    detect_and_crop_document,
    enhance_contrast,
    normalize_dpi,
)

%matplotlib inline
print('Imports successful')

## 1. Load a Sample Image

We'll use a sample image from the test suite. If you have your own medical
document scan, update the `IMAGE_PATH` variable below.

In [ ]:
# Point this to any image you want to test with
IMAGE_PATH = str(ROOT / 'tests' / 'sample_images' / 'sample.png')

# If the sample doesn't exist, create a synthetic test image
if not Path(IMAGE_PATH).exists():
    print(f'Sample image not found at {IMAGE_PATH}')
    print('Creating a synthetic test image for demonstration...')
    synthetic = np.ones((800, 600, 3), dtype=np.uint8) * 240
    cv2.putText(synthetic, 'LAB REPORT', (150, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 2)
    cv2.putText(synthetic, 'ALT: 45 U/L', (150, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    cv2.putText(synthetic, 'AST: 32 U/L', (150, 260), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    IMAGE_PATH = str(ROOT / 'notebooks' / '_sample_synthetic.png')
    cv2.imwrite(IMAGE_PATH, synthetic)

original = cv2.imread(IMAGE_PATH)
print(f'Image shape: {original.shape}')

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Document Detection & Cropping

The `detect_and_crop_document` function uses Canny edge detection and contour
finding to locate the document boundary, then applies a four-point perspective
transform to produce a top-down, cropped view.

This is critical for photos taken at an angle (e.g., phone photos of lab
reports) where the document is not aligned with the image frame.

In [ ]:
cropped = detect_and_crop_document(original)

if cropped is not None and cropped.size > 0:
    print(f'Cropped shape: {cropped.shape}')
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Before: Original')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
    axes[1].set_title('After: Document Cropped')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No document boundary detected -- using original image.')
    cropped = original

## 3. Contrast Enhancement (CLAHE)

CLAHE (Contrast Limited Adaptive Histogram Equalization) enhances local
contrast without amplifying noise. This is especially useful for:

- Low-quality phone photos with uneven lighting
- Faded printed documents
- Scans with shadows or glare

In [ ]:
enhanced = enhance_contrast(cropped)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
axes[0].set_title('Before: Cropped')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))
axes[1].set_title('After: CLAHE Contrast Enhancement')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. DPI Normalization

The `normalize_dpi` function rescales the image to a standard 300 DPI. This
ensures consistent text size across different input resolutions, which is
important for OCR engines that are sensitive to text scale.

In [ ]:
normalized = normalize_dpi(enhanced)

print(f'Before normalization: {enhanced.shape}')
print(f'After normalization:  {normalized.shape}')

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(normalized, cv2.COLOR_BGR2RGB))
plt.title('After DPI Normalization (300 DPI)')
plt.axis('off')
plt.show()

## 5. Deskew Correction

The `deskew` library estimates the skew angle of the document and rotates it
back to horizontal. This is essential for:

- Table structure detection (PaddleOCR PP-Structure)
- Line segmentation in OCR engines
- Column detection in multi-column layouts

We'll apply the deskew function manually to show the angle estimation.

In [ ]:
try:
    from deskew import determine_skew
    from skimage.transform import rotate

    gray = cv2.cvtColor(normalized, cv2.COLOR_BGR2GRAY)
    skew_angle = determine_skew(gray)
    print(f'Detected skew angle: {skew_angle:.2f} degrees')

    # Rotate to correct the skew
    deskewed = rotate(normalized, skew_angle, resize=True, mode='edge', preserve_range=True).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(cv2.cvtColor(normalized, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Before (skew: {skew_angle:.2f} deg)')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(deskewed, cv2.COLOR_BGR2RGB))
    axes[1].set_title('After Deskew')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('deskew library not installed. Install with: pip install deskew>=1.5.0')
    print('Skipping deskew demonstration.')
    deskewed = normalized

## 6. Full Preprocessing Pipeline

Now let's run the complete preprocessing pipeline using the `preprocess()`
function from `backend/preprocessing/pipeline.py`. This applies all steps in
sequence and returns the preprocessed image along with metadata about which
transformations were applied.

In [ ]:
result = preprocess(IMAGE_PATH)

print('Transformations applied:')
for t in result['transformations_applied']:
    print(f'  - {t}')

print(f'\nQuality metrics (before): {result["quality_metrics_before"]}')
print(f'Quality metrics (after):  {result["quality_metrics_after"]}')

preprocessed = result['preprocessed_image']
print(f'\nFinal preprocessed image shape: {preprocessed.shape}')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Input')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(preprocessed, cv2.COLOR_BGR2RGB))
axes[1].set_title('Fully Preprocessed')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Summary

In this notebook we explored the four preprocessing stages:

| Step | Function | Purpose |
|---|---|---|
| Document Crop | `detect_and_crop_document()` | Isolate document from background |
| Contrast Enhancement | `enhance_contrast()` | CLAHE for text legibility |
| DPI Normalization | `normalize_dpi()` | Standardize to 300 DPI |
| Deskew | `determine_skew()` + rotate | Correct rotation angle |

The `preprocess()` function in `backend/preprocessing/pipeline.py` chains
these together and returns the preprocessed image plus metadata. This
preprocessed image is then passed to the classifier and OCR stages.